In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

from utils import config_ml
from src.core.model import pack_all_data_for_ml_models, sliding_windows_cross_validating

from utils.utils import connection

import pandas as pd
import os
from tempfile import NamedTemporaryFile

import time
import logging
from datetime import timedelta

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [3]:
CONFIG = {
    "experiment_name": "stock_prediction_final",
    "run_name": "xgb_prd_model",
    "tracking_uri": "http://localhost:5001",
    "seed": 42,
    "top_n_features": 500,
    "data_description": "Данные: макрофакторы, тикерные данные, TPulse. Целевая переменная: target (регрессия). Данные зафиксированы как артефакт.",
    "train_period": config_ml.TRAIN_PERIOD,   # например, 180 дней
    "val_period": config_ml.VAL_PERIOD,       # 60 дней
    "test_period": config_ml.TEST_PERIOD,     # 30 дней
    "step": config_ml.STEP                    # 30 дней
}

In [4]:
companies = pd.read_sql("SELECT * FROM companies", connection())
tickers = companies["ticker"].tolist()
left_dates = ["2025-07-01", "2026-01-12", "2026-01-31"]
right_dates = ["2025-09-01", "2026-03-12", "2026-03-31"]

In [5]:
def save_pickle_atomic(df, path, compression='gzip'):
    # сохраняем во временный файл в той же папке, затем переименовываем
    folder = os.path.dirname(path) or '.'
    with NamedTemporaryFile(dir=folder, delete=False) as tmp:
        tmp_path = tmp.name
    df.to_pickle(tmp_path, compression=compression)
    os.replace(tmp_path, path)

In [7]:
for ticker in tickers:
    ticker_start = time.time()
    logging.info("START ticker %s", ticker)

    # если хотите показывать прогресс по внутренним циклам
    inner_count = 0
    inner_start = None

    for l, r in zip(left_dates, right_dates):
        if inner_start is None:
            inner_start = time.time()
        inner_count += 1

        # обработка одного интервала
        interval_start = time.time()
        data = pack_all_data_for_ml_models(ticker, l, r, connection())
        df = data[~data["target"].isnull()].sort_values("dt")
        
        folder = f"data/processed/{ticker}"
        os.makedirs(folder, exist_ok=True)
        
        save_pickle_atomic(df, f'data/processed/{ticker}/{l}_{r}.pkl.gz')
        
        interval_elapsed = time.time() - interval_start

        logging.info(
            "%s interval %d/%d (%s to %s) done in %s",
            ticker,
            inner_count,
            len(left_dates),
            l,
            r,
            str(timedelta(seconds=int(interval_elapsed)))
        )

    ticker_elapsed = time.time() - ticker_start
    logging.info("END ticker %s — total time %s", ticker, str(timedelta(seconds=int(ticker_elapsed))))

2026-06-02 21:26:01,010 INFO: START ticker SBER
2026-06-02 21:28:38,510 INFO: SBER interval 1/3 (2025-07-01 to 2025-09-01) done in 0:02:37
2026-06-02 21:30:13,826 INFO: SBER interval 2/3 (2026-01-12 to 2026-03-12) done in 0:01:35
2026-06-02 21:32:02,629 INFO: SBER interval 3/3 (2026-01-31 to 2026-03-31) done in 0:01:48
2026-06-02 21:32:02,630 INFO: END ticker SBER — total time 0:06:01
2026-06-02 21:32:02,630 INFO: START ticker SBERP
2026-06-02 21:33:04,054 INFO: SBERP interval 1/3 (2025-07-01 to 2025-09-01) done in 0:01:01
2026-06-02 21:33:32,794 INFO: SBERP interval 2/3 (2026-01-12 to 2026-03-12) done in 0:00:28
2026-06-02 21:34:03,610 INFO: SBERP interval 3/3 (2026-01-31 to 2026-03-31) done in 0:00:30
2026-06-02 21:34:03,611 INFO: END ticker SBERP — total time 0:02:00
2026-06-02 21:34:03,611 INFO: START ticker VTBR
2026-06-02 21:37:13,214 INFO: VTBR interval 1/3 (2025-07-01 to 2025-09-01) done in 0:03:09
2026-06-02 21:38:26,012 INFO: VTBR interval 2/3 (2026-01-12 to 2026-03-12) done 